# Fraud Mitigation Agent · 03 Context Tools

Separamos el contexto del cliente de la transacción actual.


In [ ]:
import os, sys, subprocess, json
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Walk up from the notebook's cwd until a src/fraud_mitigation_agent is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "fraud_mitigation_agent").exists():
            return candidate
    return start


# Cambia esta URL si vas a trabajar sobre tu propio fork.
REPO_URL = "https://github.com/rubencadur/fraud-mitigation-agent-workshop"

if "REPO_DIR" not in globals():
    try:
        import google.colab  # noqa: F401
        REPO_DIR = "/content/fraud-mitigation-agent-workshop"
    except ImportError:
        # Ejecución local (VS Code, Jupyter Lab, etc.): no hay clon en /content,
        # así que se ubica la raíz del repo a partir del directorio actual.
        REPO_DIR = str(_find_repo_root(Path.cwd()))

# Permite abrir este notebook de forma independiente, sin haber corrido antes
# 00_setup_workshop en la misma sesión de Colab: clona el repo e instala
# dependencias si todavía no están disponibles en este runtime.
if REPO_URL and not (Path(REPO_DIR) / "src").exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymongo[srv]", "numpy", "pandas", "python-dotenv"], check=True)

src = Path(REPO_DIR) / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

from fraud_mitigation_agent.synthetic import seed_demo_data
from fraud_mitigation_agent.local import InMemoryDB

if "db" not in globals():
    from fraud_mitigation_agent.config import Settings
    from fraud_mitigation_agent.db import get_client, get_database
    settings = Settings.from_env()
    if settings.mongodb_uri:
        client = get_client(settings.mongodb_uri)
        db = get_database(client, settings.database_name)
    else:
        db = InMemoryDB()
seed_demo_data(db, reset=False)
print("Runtime listo:", type(db).__name__)


In [ ]:
from fraud_mitigation_agent.tools.transactions import get_transaction
from fraud_mitigation_agent.tools.customer_context import get_customer_state

tx = get_transaction(db, "tx-risky-001").data
customer = get_customer_state(db, tx["customer_id"])
print(json.dumps(customer.as_dict(), indent=2, default=str))
